In [1]:
import numpy as np
import pandas as pd

# Chargement correct
Xtr = np.array(pd.read_csv('Xtr.csv', header=None, sep=',', usecols=range(3072)))
Xte = np.array(pd.read_csv('Xte.csv', header=None, sep=',', usecols=range(3072)))
Ytr = pd.read_csv('Ytr.csv', header=0, sep=',').iloc[:, 1].values
print(f" Xtr: {Xtr.shape} | Xte: {Xte.shape} | Ytr: {Ytr.shape}")

def kernel_poly(X1, X2, degree=3, c=1.0):
    return (X1 @ X2.T + c) ** degree

class KernelRidgeClassifier:
    def __init__(self, kernel_func, lam=0.1):
        self.kernel_func = kernel_func
        self.lam = lam

    def fit(self, X, y):
        self.X_train = X
        n = len(y)
        Y = np.zeros((n, 10))
        for i, label in enumerate(y):
            Y[i, label] = 1
        print("Calcul kernel...")
        K = self.kernel_func(X, X)
        print("Résolution...")
        self.alpha = np.linalg.solve(K + self.lam * np.eye(n), Y)
        print(" Entraînement terminé !")

    def predict(self, X):
        return np.argmax(self.kernel_func(X, self.X_train) @ self.alpha, axis=1)

def save_submission(predictions, filename='Yte_pred.csv'):
    df = pd.DataFrame({'Prediction': predictions})
    df.index += 1
    df.to_csv(filename, index_label='Id')
    print(f" Sauvegardé : {filename}")

 Xtr: (5000, 3072) | Xte: (2000, 3072) | Ytr: (5000,)


In [2]:
def compute_color_hist(images, n_bins=16, img_size=32):
    n = len(images)
    features = []
    for idx in range(n):
        if idx % 500 == 0:
            print(f"  ColorHist: {idx}/{n}")
        img = images[idx].reshape(3, img_size, img_size)
        hists = []
        for c in range(3):
            h, _ = np.histogram(img[c], bins=n_bins, range=(-0.5, 0.5))
            hists.append(h)
        half = img_size // 2
        for qi in range(2):
            for qj in range(2):
                patch = img[:, qi*half:(qi+1)*half, qj*half:(qj+1)*half]
                for c in range(3):
                    h, _ = np.histogram(patch[c], bins=n_bins, range=(-0.5, 0.5))
                    hists.append(h)
        feat = np.concatenate(hists).astype(float)
        feat /= (np.linalg.norm(feat) + 1e-6)
        features.append(feat)
    return np.array(features)

def compute_hog_block(images, img_size=32, cell_size=4, n_bins=9):
    n = len(images)
    features = []
    n_cells = img_size // cell_size
    for idx in range(n):
        if idx % 500 == 0:
            print(f"  HOG: {idx}/{n}")
        img = images[idx].reshape(3, img_size, img_size)
        all_hogs = []
        for c in range(3):
            gray = img[c]
            gx = np.zeros_like(gray)
            gy = np.zeros_like(gray)
            gx[:, 1:-1] = gray[:, 2:] - gray[:, :-2]
            gy[1:-1, :] = gray[2:, :] - gray[:-2, :]
            magnitude = np.sqrt(gx**2 + gy**2)
            angle = np.arctan2(gy, gx) % np.pi
            cells = np.zeros((n_cells, n_cells, n_bins))
            for i in range(n_cells):
                for j in range(n_cells):
                    mag_c = magnitude[i*cell_size:(i+1)*cell_size, j*cell_size:(j+1)*cell_size]
                    ang_c = angle[i*cell_size:(i+1)*cell_size, j*cell_size:(j+1)*cell_size]
                    cells[i,j], _ = np.histogram(ang_c, bins=n_bins, range=(0, np.pi), weights=mag_c)
            block_feat = []
            for i in range(n_cells - 1):
                for j in range(n_cells - 1):
                    block = cells[i:i+2, j:j+2].flatten()
                    block /= (np.linalg.norm(block) + 1e-6)
                    block_feat.append(block)
            all_hogs.append(np.concatenate(block_feat))
        feat = np.concatenate(all_hogs)
        feat /= (np.linalg.norm(feat) + 1e-6)
        features.append(feat)
    return np.array(features)

print("HOG block sur Xtr..."); Xtr_hog_block = compute_hog_block(Xtr)
print("HOG block sur Xte..."); Xte_hog_block = compute_hog_block(Xte)
print("Color hist sur Xtr..."); Xtr_color = compute_color_hist(Xtr)
print("Color hist sur Xte..."); Xte_color = compute_color_hist(Xte)

Xtr_comb = np.hstack([Xtr_hog_block, Xtr_color])
Xte_comb = np.hstack([Xte_hog_block, Xte_color])
Xtr_comb /= (np.linalg.norm(Xtr_comb, axis=1, keepdims=True) + 1e-6)
Xte_comb /= (np.linalg.norm(Xte_comb, axis=1, keepdims=True) + 1e-6)
mean_ = Xtr_comb.mean(axis=0); std_ = Xtr_comb.std(axis=0) + 1e-8
Xtr_scaled = (Xtr_comb - mean_) / std_
Xte_scaled = (Xte_comb - mean_) / std_
print(f" Features prêtes: {Xtr_scaled.shape}")

HOG block sur Xtr...
  HOG: 0/5000
  HOG: 500/5000
  HOG: 1000/5000
  HOG: 1500/5000
  HOG: 2000/5000
  HOG: 2500/5000
  HOG: 3000/5000
  HOG: 3500/5000
  HOG: 4000/5000
  HOG: 4500/5000
HOG block sur Xte...
  HOG: 0/2000
  HOG: 500/2000
  HOG: 1000/2000
  HOG: 1500/2000
Color hist sur Xtr...
  ColorHist: 0/5000
  ColorHist: 500/5000
  ColorHist: 1000/5000
  ColorHist: 1500/5000
  ColorHist: 2000/5000
  ColorHist: 2500/5000
  ColorHist: 3000/5000
  ColorHist: 3500/5000
  ColorHist: 4000/5000
  ColorHist: 4500/5000
Color hist sur Xte...
  ColorHist: 0/2000
  ColorHist: 500/2000
  ColorHist: 1000/2000
  ColorHist: 1500/2000
 Features prêtes: (5000, 5532)


In [3]:
kfunc = lambda X1, X2: kernel_poly(X1, X2, degree=3)
clf = KernelRidgeClassifier(kernel_func=kfunc, lam=10000)
clf.fit(Xtr_scaled, Ytr)
Yte = clf.predict(Xte_scaled)
save_submission(Yte, 'Yte_block.csv')

Calcul kernel...
Résolution...
 Entraînement terminé !
 Sauvegardé : Yte_block.csv


In [4]:
# Test kernel polynomial degree=3 avec plus de lambda
N_train, N_val = 4500, 500
print("Dernier test:")
for lam in [5000, 8000, 10000, 15000]:
    kfunc = lambda X1, X2: kernel_poly(X1, X2, degree=3)
    clf = KernelRidgeClassifier(kernel_func=kfunc, lam=lam)
    clf.fit(Xtr_scaled[:N_train], Ytr[:N_train])
    acc_val = np.mean(clf.predict(Xtr_scaled[N_train:]) == Ytr[N_train:])
    print(f"  lam={lam} → Val: {acc_val:.2%}")

Dernier test:
Calcul kernel...
Résolution...
 Entraînement terminé !
  lam=5000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
  lam=8000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
  lam=10000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
  lam=15000 → Val: 63.60%


In [5]:
# Kernel polynomial deg=3 avec c différent
def kernel_poly(X1, X2, degree=3, c=0.1):
    return (X1 @ X2.T + c) ** degree

for c in [0.1, 0.5, 2.0, 5.0]:
    for lam in [5000, 10000]:
        kfunc = lambda X1, X2, c=c: kernel_poly(X1, X2, degree=3, c=c)
        clf = KernelRidgeClassifier(kernel_func=kfunc, lam=lam)
        clf.fit(Xtr_scaled[:4500], Ytr[:4500])
        acc = np.mean(clf.predict(Xtr_scaled[4500:]) == Ytr[4500:])
        print(f"c={c}, lam={lam} → Val: {acc:.2%}")

Calcul kernel...
Résolution...
 Entraînement terminé !
c=0.1, lam=5000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=0.1, lam=10000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=0.5, lam=5000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=0.5, lam=10000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=2.0, lam=5000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=2.0, lam=10000 → Val: 63.60%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=5.0, lam=5000 → Val: 63.40%
Calcul kernel...
Résolution...
 Entraînement terminé !
c=5.0, lam=10000 → Val: 63.40%
